# Operaciones Dataframes

Las listas de Row como se usan en el notebook NO se usan actualmente en código de producción. Es código legacy de Spark 1.x/2.x.

Por qué no se usan:
* Rendimiento pobre - Spark tiene que inferir schema
* Memoria ineficiente - Objetos Python vs datos nativos
* Debugging difícil - Estructuras anidadas complejas
* No escalable - Problemas con datasets grandes

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("dataframes").master("spark://spark-master:7077").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/09/02 00:12:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Imports

In [2]:
# Creacion de  DataFrames
# Se importan todas las clases del modulo sql
from pyspark.sql import *

# Ejemplo - Departments y Employees

# Se crean los Departments, Row es un objeto de fila genérico 
# con una colección ordenada de campos a los que se puede acceder 
# mediante un índice ordinal.
# Row también se llama Catalyst Row. La fila puede tener un esquema opcional.
department1 = Row(id='123456', name='Computer Science')
department2 = Row(id='789012', name='Mechanical Engineering')
department3 = Row(id='345678', name='Theater and Drama')
department4 = Row(id='901234', name='Indoor Recreation')

# Se crean los Employees
Employee = Row("firstName", "lastName", "email", "salary")
employee1 = Employee('michael', 'armbrust', 'no-reply@berkeley.edu', 100000)
employee2 = Employee('xiangrui', 'meng', 'no-reply@stanford.edu', 120000)
employee3 = Employee('matei', None, 'no-reply@waterloo.edu', 140000)
employee4 = Employee(None, 'wendell', 'no-reply@berkeley.edu', 160000)
employee5 = Employee('michael', 'jackson', 'no-reply@neverla.nd', 80000)

# Se crean las instancias DepartmentWithEmployees a partir de Departments y Employees
# "employees" es una lista de Row, la lista es un objeto array
departmentWithEmployees1 = Row(department=department1, employees=[employee1, employee2])
departmentWithEmployees2 = Row(department=department2, employees=[employee3, employee4])
departmentWithEmployees3 = Row(department=department3, employees=[employee5, employee4])
departmentWithEmployees4 = Row(department=department4, employees=[employee2, employee3])

print(department1)
print(employee2)
print(departmentWithEmployees1.employees[0].email)

Row(id='123456', name='Computer Science')
Row(firstName='xiangrui', lastName='meng', email='no-reply@stanford.edu', salary=120000)
no-reply@berkeley.edu


In [3]:
# Creacion de 2 DataFrames a partir de listas de row
departmentsWithEmployeesSeq1 = [departmentWithEmployees1, departmentWithEmployees2]
df1 = spark.createDataFrame(departmentsWithEmployeesSeq1)

df1.show(truncate=False)

departmentsWithEmployeesSeq2 = [departmentWithEmployees3, departmentWithEmployees4]
df2 = spark.createDataFrame(departmentsWithEmployeesSeq2)

df2.show(truncate=False)

+--------------------------------+-----------------------------------------------------------------------------------------------------+
|department                      |employees                                                                                            |
+--------------------------------+-----------------------------------------------------------------------------------------------------+
|{123456, Computer Science}      |[{michael, armbrust, no-reply@berkeley.edu, 100000}, {xiangrui, meng, no-reply@stanford.edu, 120000}]|
|{789012, Mechanical Engineering}|[{matei, null, no-reply@waterloo.edu, 140000}, {null, wendell, no-reply@berkeley.edu, 160000}]       |
+--------------------------------+-----------------------------------------------------------------------------------------------------+

+---------------------------+------------------------------------------------------------------------------------------------+
|department                 |employees            

In [4]:
# Se concatenan ambos DataFrames
unionDF = df1.union(df2)
unionDF.show(truncate=False)

+--------------------------------+-----------------------------------------------------------------------------------------------------+
|department                      |employees                                                                                            |
+--------------------------------+-----------------------------------------------------------------------------------------------------+
|{123456, Computer Science}      |[{michael, armbrust, no-reply@berkeley.edu, 100000}, {xiangrui, meng, no-reply@stanford.edu, 120000}]|
|{789012, Mechanical Engineering}|[{matei, null, no-reply@waterloo.edu, 140000}, {null, wendell, no-reply@berkeley.edu, 160000}]       |
|{345678, Theater and Drama}     |[{michael, jackson, no-reply@neverla.nd, 80000}, {null, wendell, no-reply@berkeley.edu, 160000}]     |
|{901234, Indoor Recreation}     |[{xiangrui, meng, no-reply@stanford.edu, 120000}, {matei, null, no-reply@waterloo.edu, 140000}]      |
+--------------------------------+-------

In [6]:
unionDF.printSchema()

root
 |-- department: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- employees: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- firstName: string (nullable = true)
 |    |    |-- lastName: string (nullable = true)
 |    |    |-- email: string (nullable = true)
 |    |    |-- salary: long (nullable = true)



In [5]:
# Con Explode se convierte la columna employees que es un array de listas, a un formato tabular
from pyspark.sql.functions import explode

explodeDF = unionDF.select(explode("employees").alias("e"))
flattenDF = explodeDF.selectExpr("e.firstName", "e.lastName", "e.email", "e.salary")

flattenDF.show(truncate=False)

+---------+--------+---------------------+------+
|firstName|lastName|email                |salary|
+---------+--------+---------------------+------+
|michael  |armbrust|no-reply@berkeley.edu|100000|
|xiangrui |meng    |no-reply@stanford.edu|120000|
|matei    |null    |no-reply@waterloo.edu|140000|
|null     |wendell |no-reply@berkeley.edu|160000|
|michael  |jackson |no-reply@neverla.nd  |80000 |
|null     |wendell |no-reply@berkeley.edu|160000|
|xiangrui |meng    |no-reply@stanford.edu|120000|
|matei    |null    |no-reply@waterloo.edu|140000|
+---------+--------+---------------------+------+



In [11]:
# Se usa filter() para retornar aquellas filas que coincidan
# Se usa sort() para ordenar
filterDF = flattenDF.filter(flattenDF.firstName == "xiangrui").sort(flattenDF.lastName)

In [14]:
from pyspark.sql.functions import col, asc
# Se usa `|` en lugar de `or`
filterDF = flattenDF.filter((col("firstName") == "xiangrui") | (col("firstName") == "michael")).sort(asc("lastName"))
filterDF.show()

[Stage 10:=============================>                            (4 + 4) / 8]

+---------+--------+--------------------+------+
|firstName|lastName|               email|salary|
+---------+--------+--------------------+------+
|  michael|armbrust|no-reply@berkeley...|100000|
|  michael| jackson| no-reply@neverla.nd| 80000|
| xiangrui|    meng|no-reply@stanford...|120000|
| xiangrui|    meng|no-reply@stanford...|120000|
+---------+--------+--------------------+------+



In [15]:
# La sentencia where() es equivalente a filter()
whereDF = flattenDF.where((col("firstName") == "xiangrui") | (col("firstName") == "michael")).sort(asc("lastName"))
whereDF.show()

+---------+--------+--------------------+------+
|firstName|lastName|               email|salary|
+---------+--------+--------------------+------+
|  michael|armbrust|no-reply@berkeley...|100000|
|  michael| jackson| no-reply@neverla.nd| 80000|
| xiangrui|    meng|no-reply@stanford...|120000|
| xiangrui|    meng|no-reply@stanford...|120000|
+---------+--------+--------------------+------+



In [18]:
# Se reemplazan los valores nulos con '--' usando el método fillna()
nonNullDF = flattenDF.fillna("--")
nonNullDF.show(truncate=False)

+---------+--------+---------------------+------+
|firstName|lastName|email                |salary|
+---------+--------+---------------------+------+
|michael  |armbrust|no-reply@berkeley.edu|100000|
|xiangrui |meng    |no-reply@stanford.edu|120000|
|matei    |--      |no-reply@waterloo.edu|140000|
|--       |wendell |no-reply@berkeley.edu|160000|
|michael  |jackson |no-reply@neverla.nd  |80000 |
|--       |wendell |no-reply@berkeley.edu|160000|
|xiangrui |meng    |no-reply@stanford.edu|120000|
|matei    |--      |no-reply@waterloo.edu|140000|
+---------+--------+---------------------+------+



In [ ]:
# Se visualizan las filas que tengan valores nulos filtrando con el método isNull()
filterNonNullDF = flattenDF.filter(col("firstName").isNull() | col("lastName").isNull()).sort("email")
display(filterNonNullDF)

In [ ]:
# Se importa countDistinct para aplicar la agregacion con el método agg()
from pyspark.sql.functions import countDistinct

countDistinctDF = nonNullDF.select("firstName", "lastName")\
  .groupBy("firstName")\
  .agg(countDistinct("lastName").alias("distinct_last_names"))

countDistinctDF.show(truncate=False)

In [ ]:
# Se comparan los planes de ejecución entre la consulta sobre el DataFrame y sobre la tabla temporal con SQL
countDistinctDF.explain()

In [ ]:
# Se crea la tabla temporal, a partir del DataFrame
nonNullDF.createOrReplaceTempView("databricks_df_example")

# Se genera la misma consulta sobre la tabla temporal con SQL para luego retronar el plan de ejecución
countDistinctDF_sql = spark.sql('''
  SELECT firstName, count(distinct lastName) AS distinct_last_names
  FROM databricks_df_example
  GROUP BY firstName
''')

countDistinctDF_sql.explain()

In [ ]:
# Se aplica la agregacion de suma sobre 'salary'
salarySumDF = nonNullDF.agg({"salary" : "sum"})
salarySumDF.show(truncate=False)

In [ ]:
type(nonNullDF.salary)

In [ ]:
# Se imprimen las métricas estadísticas sobre la columna 'salary'
nonNullDF.describe("salary").show()

In [19]:
# DataFrame FAQs
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Create the data
data = [
    (1, "2015-10-14 00:00:00", "2015-09-14 00:00:00", "CA-SF"),
    (2, "2015-10-15 01:00:20", "2015-08-14 00:00:00", "CA-SD"),
    (3, "2015-10-16 02:30:00", "2015-01-14 00:00:00", "NY-NY"),
    (4, "2015-10-17 03:00:20", "2015-02-14 00:00:00", "NY-NY"),
    (5, "2015-10-18 04:30:00", "2014-04-14 00:00:00", "CA-SD")
]

# Define schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("end_date", StringType(), True),
    StructField("start_date", StringType(), True),
    StructField("location", StringType(), True)
])

# Create DataFrame
df = spark.createDataFrame(data, schema)
df.show()

+---+-------------------+-------------------+--------+
| id|           end_date|         start_date|location|
+---+-------------------+-------------------+--------+
|  1|2015-10-14 00:00:00|2015-09-14 00:00:00|   CA-SF|
|  2|2015-10-15 01:00:20|2015-08-14 00:00:00|   CA-SD|
|  3|2015-10-16 02:30:00|2015-01-14 00:00:00|   NY-NY|
|  4|2015-10-17 03:00:20|2015-02-14 00:00:00|   NY-NY|
|  5|2015-10-18 04:30:00|2014-04-14 00:00:00|   CA-SD|
+---+-------------------+-------------------+--------+



In [20]:
# En lugar de registrar una UDF, se usan las funciones integradas para realizar operaciones en las columnas.
# Esto proporcionará una mejora en el rendimiento a medida que las incorporaciones se compilan y se ejecutan en la JVM de la plataforma.

# Se convierte a tipo Date
df = df.withColumn('date', F.to_date(df.end_date))

# Se parsea la columna date, que contiene fecha y hora, para obtener solamente fecha
df = df.withColumn('date_only', F.regexp_replace(df.end_date,' (\d+)[:](\d+)[:](\d+).*$', ''))

# Split subdivide el valor con el delimitador '-' que se explicita, lo que genera un array de los elementos obtenidos
# En el ejemplo, se queda con el indice 1
df = df.withColumn('city', F.split(df.location, '-')[1])

# Se obtiene la diferencia en días de dos fechas
df = df.withColumn('date_diff', F.datediff(F.to_date(df.end_date), F.to_date(df.start_date)))

In [21]:
df.createOrReplaceTempView("sample_df")
spark.sql("select * from sample_df").show(truncate=False)

[Stage 21:===================>                                      (1 + 2) / 3]

+---+-------------------+-------------------+--------+----------+----------+----+---------+
|id |end_date           |start_date         |location|date      |date_only |city|date_diff|
+---+-------------------+-------------------+--------+----------+----------+----+---------+
|1  |2015-10-14 00:00:00|2015-09-14 00:00:00|CA-SF   |2015-10-14|2015-10-14|SF  |30       |
|2  |2015-10-15 01:00:20|2015-08-14 00:00:00|CA-SD   |2015-10-15|2015-10-15|SD  |62       |
|3  |2015-10-16 02:30:00|2015-01-14 00:00:00|NY-NY   |2015-10-16|2015-10-16|NY  |275      |
|4  |2015-10-17 03:00:20|2015-02-14 00:00:00|NY-NY   |2015-10-17|2015-10-17|NY  |245      |
|5  |2015-10-18 04:30:00|2014-04-14 00:00:00|CA-SD   |2015-10-18|2015-10-18|SD  |552      |
+---+-------------------+-------------------+--------+----------+----------+----+---------+



25/09/02 00:23:04 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
25/09/02 00:23:04 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:218)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:923)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:154)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:262)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:169)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce